"""
========================================================================
NOTEBOOK 03: GEMINI CROSS-MODEL ROBUSTNESS PROBE
========================================================================

Tests whether the prompt-engineerability hierarchy and topic-non-responsive
finding generalize beyond OpenAI to Google Gemini 2.5 Flash.

Design:
  - Same 20 headlines as Notebook 02 (sample_id 0-19)
  - 2 prompts: A (default NYT-style) + E (frame + anti-brochure)
  - 3 reps per (headline, prompt) cell
  - 120 Gemini API calls total

Why A and E only:
  A = default state (test if positive offset replicates)
  E = best intervention (test if anti-brochure recovers gap)
  Skip B and D — those were mechanism probes for OpenAI specifically.
  Cross-model robustness needs the two endpoints, not the full ladder.

Expected outcomes:
  - If Gemini A also positive offset and Gemini E reduces it:
    → hierarchy generalizes; paper claim strengthened
  - If Gemini A is already close to NYT:
    → finding is OpenAI-specific (still publishable but reframed)
  - If Gemini E doesn't help:
    → prompt-engineerability is OpenAI-specific
  All three are paper-worthy.

Outputs (data/generations/, data/results/):
  - generations_gemini.csv
  - axis2_gemini.csv
  - results_crossmodel_summary.csv
  - results_crossmodel_paired.csv

Prereqs:
  - Notebook 01 has been run successfully (NYT baseline + sample exist)
  - Notebook 02 has been run (we'll merge with its GPT axis2 data)
  - GOOGLE_API_KEY is set
"""


In [ ]:
# Set your Google API key in the environment BEFORE running (do NOT hardcode):
#   export GOOGLE_API_KEY="AIza..."
import os
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your environment first."

In [ ]:

# =============================================================================
# CELL 1: Setup & paths (relative to notebooks/, same as Notebook 01/02)
# =============================================================================

import os
import re
import json
import gc
import time
import warnings
import urllib.request
import urllib.error
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
warnings.filterwarnings("ignore", message="Token indices sequence length")

# Paths relative to this notebook (notebooks/), same convention as Notebook 01/02
ROOT = Path("..").resolve()
PROCESSED = ROOT / "data" / "processed"
GENERATIONS = ROOT / "data" / "generations"
RESULTS = ROOT / "data" / "results"
GENERATIONS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# Verify everything exists
assert ROOT.exists(), f"ROOT not found: {ROOT}"
for f in ["experiment_sample.csv", "nyt_baseline_axis2.csv"]:
    assert (PROCESSED / f).exists(), f"Missing Notebook 01 output: {f}"
assert (RESULTS / "axis2_gpt.csv").exists(), \
    "Missing Notebook 02 output: axis2_gpt.csv (run Notebook 02 first)"

print(f"ROOT:        {ROOT}")
print(f"PROCESSED:   {PROCESSED}")
print(f"GENERATIONS: {GENERATIONS}")
print(f"RESULTS:     {RESULTS}")
print("✓ All prereq files present.")

In [ ]:

# =============================================================================
# CELL 2: Load Notebook 01/02 outputs
# =============================================================================

sampled = pd.read_csv(PROCESSED / "experiment_sample.csv")
sampled["pub_date"] = pd.to_datetime(sampled["pub_date"])
nyt_axis2 = pd.read_csv(PROCESSED / "nyt_baseline_axis2.csv")
gpt_axis2 = pd.read_csv(RESULTS / "axis2_gpt.csv")

print(f"Sample:    {len(sampled)} headlines")
print(f"NYT axis2: {len(nyt_axis2)} rows")
print(f"GPT axis2: {len(gpt_axis2)} rows")


In [ ]:

# =============================================================================
# CELL 3: Define prompts A and E (must match Notebook 02 exactly)
# =============================================================================

SYSTEM_A = (
    "You are a staff journalist writing for The New York Times. "
    "You write in standard NYT register: clear lede, supporting paragraphs "
    "with context and attribution, sourcing to named or general authorities "
    "(e.g., 'analysts said', 'according to company filings'). "
    "Do not fabricate direct quotes from named individuals you cannot verify. "
    "Write only the article body — no headline, no byline, no editor's note."
)
PROMPT_A = """SECTION: {section} desk
DATE OF PUBLICATION: {date}
HEADLINE: "{headline}"

Write the complete news article as it would have appeared in print on this date, in this section, under this headline. Target length is roughly 900 to 1,300 words, with a proper lede and several supporting paragraphs. Write from the knowledge and perspective available at the time of publication.

Begin the article now."""

SYSTEM_E = SYSTEM_A
PROMPT_E = """SECTION: {section} desk
DATE OF PUBLICATION: {date}
HEADLINE: "{headline}"

Write the complete news article as it would have appeared in print on this date, in this section, under this headline. Target length is roughly 900 to 1,300 words, with a proper lede and several supporting paragraphs. Write from the knowledge and perspective available at the time of publication.

Use the section-specific frame as the primary organizing logic of the article, while adapting it to the specific headline:
- For Business desk articles, foreground firms, markets, investors, regulation, supply chains, corporate strategy, economic uncertainty, and commercial or technological risk.
- For World desk articles, foreground states, diplomacy, security, governance, rights, sovereignty, international institutions, and geopolitical consequences.

Additional style constraints:
- Avoid promotional, celebratory, or corporate brochure-like language.
- Do not frame the story as a corporate announcement, market opportunity, or institutional success story unless the headline explicitly requires it.
- Lead with concrete stakes, tensions, constraints, risks, or consequences rather than generic background or scene-setting.
- Assume readers already know the basic context about the main entities involved; avoid over-explaining.
- Use sober, direct, plain language.

Begin the article now."""

GEMINI_PROMPTS = {
    "A": (SYSTEM_A, PROMPT_A),
    "E": (SYSTEM_E, PROMPT_E),
}



In [ ]:

# =============================================================================
# CELL 4: Gemini API caller
# =============================================================================

assert os.environ.get("GOOGLE_API_KEY"), \
    "GOOGLE_API_KEY not set. Run: os.environ['GOOGLE_API_KEY'] = 'AIza...'"

GEMINI_MODEL = "gemini-2.5-flash"


def call_gemini(system, user, model=GEMINI_MODEL,
                temperature=0.7, max_output_tokens=3200,
                retries=4, sleep_seconds=5):
    api_key = os.environ["GOOGLE_API_KEY"].strip()
    url = (
        f"https://generativelanguage.googleapis.com/v1beta/models/"
        f"{model}:generateContent?key={api_key}"
    )
    body = {
        "system_instruction": {"parts": [{"text": system}]},
        "contents": [{"role": "user", "parts": [{"text": user}]}],
        "generationConfig": {
            "temperature": temperature,
            "maxOutputTokens": max_output_tokens,
        },
    }
    body_bytes = json.dumps(body).encode("utf-8")

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(
                url, data=body_bytes,
                headers={"Content-Type": "application/json"},
                method="POST",
            )
            with urllib.request.urlopen(req, timeout=180) as resp:
                data = json.loads(resp.read().decode("utf-8"))

            candidates = data.get("candidates", [])
            if not candidates:
                raise RuntimeError(f"No candidates: {data}")
            content = candidates[0].get("content", {})
            parts = content.get("parts", [])
            if not parts:
                fr = candidates[0].get("finishReason", "UNKNOWN")
                raise RuntimeError(f"Empty parts. finishReason={fr}")
            text = "".join(p.get("text", "") for p in parts).strip()
            if not text:
                raise RuntimeError("Empty text")
            return text
        except urllib.error.HTTPError as e:
            try:
                body_resp = e.read()[:300]
            except Exception:
                body_resp = b""
            last_err = f"HTTP {e.code}: {body_resp!r}"
            print(f"  [retry {attempt}/{retries}] {last_err}")
            time.sleep(sleep_seconds * attempt)
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
            print(f"  [retry {attempt}/{retries}] {last_err}")
            time.sleep(sleep_seconds * attempt)
    raise RuntimeError(f"Gemini failed after {retries} attempts: {last_err}")


# Smoke test
print("Testing Gemini connectivity...")
test_resp = call_gemini(
    "You are a helpful assistant.",
    "Reply with exactly the word: pong",
    temperature=0,
    max_output_tokens=20,
)
print(f"Response: {test_resp}")


In [ ]:

# =============================================================================
# CELL 5: Generate Gemini articles (len(sampled) x 2 prompts x REPS) with checkpoint
# =============================================================================

REPS = 3
GEN_FILE = GENERATIONS / "generations_gemini.csv"

if GEN_FILE.exists():
    df_gem = pd.read_csv(GEN_FILE)
    done = set(zip(df_gem["sample_id"], df_gem["rep"], df_gem["prompt_version"]))
    rows = df_gem.to_dict("records")
    print(f"Resuming: {len(done)} Gemini generations already saved.")
else:
    rows = []
    done = set()

target = len(GEMINI_PROMPTS) * len(sampled) * REPS  # prompts x headlines x reps
print(f"Target: {target}. Done: {len(done)}. To do: {target - len(done)}")

saved_since = 0
SAVE_EVERY = 10

for prompt_name, (sys_msg, user_template) in GEMINI_PROMPTS.items():
    for _, r in sampled.iterrows():
        sid = r["sample_id"]
        section = "Business" if r["news_desk"] == "Business" else "World"
        date_str = r["pub_date"].strftime("%B %d, %Y").replace(" 0", " ")
        prompt = user_template.format(
            section=section, date=date_str, headline=r["headline"],
        )
        for rep in range(REPS):
            key = (sid, rep, prompt_name)
            if key in done:
                continue
            print(f"[{prompt_name}] sid={sid} {section[:3]} rep={rep+1}/{REPS}  "
                  f"{r['headline'][:55]}...")
            try:
                text = call_gemini(sys_msg, prompt)
                wc = len(text.split())
                print(f"    → {wc} words")
            except Exception as e:
                print(f"  !! {type(e).__name__}: {e}")
                text = f"[ERROR: {type(e).__name__}]"
                wc = 0

            rows.append({
                "sample_id": sid,
                "rep": rep,
                "prompt_version": prompt_name,
                "headline": r["headline"],
                "news_desk": r["news_desk"],
                "section_prompted": section,
                "pub_date": str(r["pub_date"]),
                "generated_full_text": text,
                "generated_word_count": wc,
                "model": GEMINI_MODEL,
            })
            saved_since += 1

            if saved_since >= SAVE_EVERY:
                pd.DataFrame(rows).to_csv(GEN_FILE, index=False)
                print(f"    ✓ checkpoint ({len(rows)} rows)")
                saved_since = 0

            time.sleep(0.5)  # Gemini free tier: respect rate limit

df_gem = pd.DataFrame(rows)
df_gem.to_csv(GEN_FILE, index=False)
print(f"\nTotal Gemini rows: {len(df_gem)}")
ok = df_gem[~df_gem["generated_full_text"].astype(str).str.startswith("[ERROR")]
print(f"Successful: {len(ok)}")
print("\nWord counts:")
print(ok.groupby(["prompt_version", "news_desk"])["generated_word_count"].describe().round(0))

In [ ]:

# =============================================================================
# CELL 6: Load DistilBERT for sentiment scoring (same as Notebook 01/02)
# =============================================================================

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
MAX_TOKENS_PER_CHUNK = 450
STRIDE = 50
BATCH_SIZE = 8

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=False)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
sentiment_model.to(device)
sentiment_model.eval()
print(f"Device: {device}")


def clear_memory():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def make_token_chunks(text, max_tokens=MAX_TOKENS_PER_CHUNK, stride=STRIDE):
    if not isinstance(text, str) or not text.strip():
        return []
    tids = tokenizer(text, add_special_tokens=False, truncation=False,
                    return_attention_mask=False)["input_ids"]
    if not tids:
        return []
    chunks, start = [], 0
    while start < len(tids):
        end = start + max_tokens
        chunks.append([tokenizer.cls_token_id] + tids[start:end] + [tokenizer.sep_token_id])
        if end >= len(tids):
            break
        start = end - stride
    return chunks


def predict_token_chunks(token_id_chunks, batch_size=BATCH_SIZE):
    if not token_id_chunks:
        return pd.DataFrame()
    rows_out = []
    for start in range(0, len(token_id_chunks), batch_size):
        batch = token_id_chunks[start:start + batch_size]
        max_len = max(len(x) for x in batch)
        pad_id = tokenizer.pad_token_id
        input_ids, attn = [], []
        for ids in batch:
            pad_n = max_len - len(ids)
            input_ids.append(ids + [pad_id] * pad_n)
            attn.append([1] * len(ids) + [0] * pad_n)
        inputs = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long).to(device),
            "attention_mask": torch.tensor(attn, dtype=torch.long).to(device),
        }
        with torch.inference_mode():
            out = sentiment_model(**inputs)
            probs = torch.softmax(out.logits, dim=-1).detach().cpu().numpy()
        for j, prob in enumerate(probs):
            rows_out.append({
                "chunk_id": start + j,
                "neg_prob": float(prob[0]),
                "pos_prob": float(prob[1]),
                "sentiment_score": float(prob[1]) - float(prob[0]),
            })
        clear_memory()
    return pd.DataFrame(rows_out)

In [ ]:

# =============================================================================
# CELL 7: Score Gemini generations on sentiment axis
# =============================================================================

axis2_rows = []
print(f"Scoring {len(df_gem)} Gemini generations on sentiment axis...")
for _, r in tqdm(df_gem.iterrows(), total=len(df_gem)):
    if str(r["generated_full_text"]).startswith("[ERROR"):
        continue
    try:
        chunks = make_token_chunks(r["generated_full_text"])
        chunk_df = predict_token_chunks(chunks)
        if len(chunk_df) > 0:
            axis2_rows.append({
                "sample_id": r["sample_id"],
                "rep": r["rep"],
                "prompt_version": r["prompt_version"],
                "news_desk": r["news_desk"],
                "model": "Gemini-2.5-Flash",
                "sentiment": chunk_df["sentiment_score"].mean(),
                "n_chunks": len(chunk_df),
                "word_count": r["generated_word_count"],
            })
    except Exception as e:
        print(f"Error sid={r['sample_id']} {r['prompt_version']} rep={r['rep']}: {e}")

axis2_gemini = pd.DataFrame(axis2_rows)
axis2_gemini.to_csv(RESULTS / "axis2_gemini.csv", index=False)
print(f"Saved: axis2_gemini.csv ({len(axis2_gemini)} rows)")

In [ ]:
# =============================================================================
# CELL 8: Cross-model comparison — A and E for both models vs NYT
# =============================================================================

# Restrict GPT to only A and E for fair comparison
gpt_AE = gpt_axis2[gpt_axis2["prompt_version"].isin(["A", "E"])].copy()
gpt_AE["model"] = "GPT-4o-mini"

# Stack all data
all_data = pd.concat([
    nyt_axis2.assign(model="NYT (real)", prompt_version="NYT", rep=-1)
        [["sample_id", "rep", "prompt_version", "news_desk", "model", "sentiment"]],
    gpt_AE[["sample_id", "rep", "prompt_version", "news_desk", "model", "sentiment"]],
    axis2_gemini[["sample_id", "rep", "prompt_version", "news_desk", "model", "sentiment"]],
], ignore_index=True)

# Headline-aggregated (average reps)
agg = all_data.groupby(
    ["sample_id", "model", "prompt_version", "news_desk"], as_index=False
)["sentiment"].mean()


# ---- Cross-model summary table ----
print("=" * 78)
print("CROSS-MODEL COMPARISON — center and gap")
print("=" * 78)
print()

summary_rows = []
for model_label in ["NYT (real)", "GPT-4o-mini", "Gemini-2.5-Flash"]:
    for v in ["NYT", "A", "E"]:
        sub = agg[(agg["model"] == model_label) & (agg["prompt_version"] == v)]
        if len(sub) == 0:
            continue
        biz = sub[sub["news_desk"] == "Business"]["sentiment"]
        foreign = sub[sub["news_desk"] == "Foreign"]["sentiment"]
        if not (len(biz) and len(foreign)):
            continue
        center = (biz.mean() + foreign.mean()) / 2
        gap = biz.mean() - foreign.mean()
        direction = ("matches paradox" if gap < 0
                     else "REVERSED" if gap > 0.01 else "FLAT")
        print(f"  {model_label:18s} {v:4s}  "
              f"Biz={biz.mean():+.4f}  For={foreign.mean():+.4f}  "
              f"Gap={gap:+.4f} ({direction})  Center={center:+.4f}")
        summary_rows.append({
            "model": model_label, "prompt": v,
            "n_biz": len(biz), "n_for": len(foreign),
            "biz_mean": biz.mean(), "for_mean": foreign.mean(),
            "center": center, "gap": gap,
        })

pd.DataFrame(summary_rows).to_csv(RESULTS / "results_crossmodel_summary.csv", index=False)
print(f"\nSaved: results_crossmodel_summary.csv")

In [ ]:

# =============================================================================
# CELL 9: Per-headline LLM-NYT deviation for both models
# =============================================================================

print("\n" + "=" * 78)
print("PER-HEADLINE DEVIATION (controls for headline charge)")
print("=" * 78)
print()

# NYT per-headline values
nyt_per = agg[agg["model"] == "NYT (real)"][["sample_id", "sentiment"]].rename(
    columns={"sentiment": "nyt_sentiment"}
)
llm_only = agg[agg["model"].isin(["GPT-4o-mini", "Gemini-2.5-Flash"])].merge(
    nyt_per, on="sample_id"
)
llm_only["deviation"] = llm_only["sentiment"] - llm_only["nyt_sentiment"]

print("Mean per-headline deviation (LLM - NYT) by model × prompt × desk:")
print()
deviation_summary = llm_only.groupby(
    ["model", "prompt_version", "news_desk"]
).agg(
    n=("deviation", "count"),
    deviation_mean=("deviation", "mean"),
    deviation_std=("deviation", "std"),
).round(4).reset_index()
print(deviation_summary.to_string(index=False))
deviation_summary.to_csv(RESULTS / "results_crossmodel_per_headline.csv", index=False)
print(f"\nSaved: results_crossmodel_per_headline.csv")

print("\nMean |deviation| (overall, lower = closer to NYT):")
abs_dev = llm_only.copy()
abs_dev["abs_deviation"] = abs_dev["deviation"].abs()
overall_abs = abs_dev.groupby(["model", "prompt_version"])["abs_deviation"].agg(
    ["mean", "std", "count"]
).round(4)
print(overall_abs)


In [ ]:

# =============================================================================
# CELL 10: Paired Wilcoxon — does each model × prompt differ from NYT?
# =============================================================================

from scipy.stats import wilcoxon

print("\n" + "=" * 78)
print("PAIRED WILCOXON — LLM vs NYT (within-headline)")
print("=" * 78)
print()

paired = []
for model_label in ["GPT-4o-mini", "Gemini-2.5-Flash"]:
    for v in ["A", "E"]:
        for desk in ["Business", "Foreign"]:
            sub = llm_only[
                (llm_only["model"] == model_label)
                & (llm_only["prompt_version"] == v)
                & (llm_only["news_desk"] == desk)
            ]
            if len(sub) < 5:
                continue
            try:
                stat, p = wilcoxon(
                    sub["sentiment"].values, sub["nyt_sentiment"].values,
                    alternative="two-sided",
                )
            except Exception:
                stat, p = np.nan, np.nan
            sig = "✓" if p < 0.05 else " "
            print(f"  {model_label:18s} {v} {desk:9s}  n={len(sub):2d}  "
                  f"LLM={sub['sentiment'].mean():+.4f}  NYT={sub['nyt_sentiment'].mean():+.4f}  "
                  f"deviation={sub['deviation'].mean():+.4f}  p={p:.4f}  {sig}")
            paired.append({
                "model": model_label, "prompt": v, "desk": desk, "n": len(sub),
                "llm_mean": sub["sentiment"].mean(),
                "nyt_mean": sub["nyt_sentiment"].mean(),
                "deviation": sub["deviation"].mean(),
                "wilcoxon_stat": stat, "p_value": p,
            })

pd.DataFrame(paired).to_csv(RESULTS / "results_crossmodel_paired.csv", index=False)
print(f"\nSaved: results_crossmodel_paired.csv")


In [ ]:


# =============================================================================
# CELL 11: Three explicit hierarchy questions for each model
# =============================================================================

print("\n" + "=" * 78)
print("HIERARCHY PRESERVATION CHECK")
print("=" * 78)


def get_stats(model_label, v):
    sub = agg[(agg["model"] == model_label) & (agg["prompt_version"] == v)]
    biz = sub[sub["news_desk"] == "Business"]["sentiment"]
    foreign = sub[sub["news_desk"] == "Foreign"]["sentiment"]
    if not (len(biz) and len(foreign)):
        return None
    return {
        "biz_mean": biz.mean(), "for_mean": foreign.mean(),
        "center": (biz.mean() + foreign.mean()) / 2,
        "gap": biz.mean() - foreign.mean(),
    }


nyt = get_stats("NYT (real)", "NYT")
print(f"\nNYT baseline: center={nyt['center']:+.4f}  gap={nyt['gap']:+.4f}")

for model_label in ["GPT-4o-mini", "Gemini-2.5-Flash"]:
    A = get_stats(model_label, "A")
    E = get_stats(model_label, "E")
    if A is None or E is None:
        continue
    print(f"\n{model_label}")
    print(f"  A:  center={A['center']:+.4f}  gap={A['gap']:+.4f}")
    print(f"  E:  center={E['center']:+.4f}  gap={E['gap']:+.4f}")

    # Q1: A center too positive vs NYT?
    q1 = A["center"] > nyt["center"] + 0.10
    print(f"  Q1: A center too positive vs NYT?  "
          f"A={A['center']:+.4f} vs NYT={nyt['center']:+.4f}  "
          f"{'YES' if q1 else 'NO'}")

    # Q2: E reduces center distance from NYT?
    a_dist = abs(A["center"] - nyt["center"])
    e_dist = abs(E["center"] - nyt["center"])
    q2 = e_dist < a_dist
    print(f"  Q2: E moves center toward NYT?  "
          f"A_dist={a_dist:.4f} → E_dist={e_dist:.4f}  "
          f"{'YES' if q2 else 'NO'}")

    # Q3: E gap closer to NYT direction?
    q3 = (E["gap"] < 0 and abs(E["gap"] - nyt["gap"]) < abs(A["gap"] - nyt["gap"]))
    print(f"  Q3: E gap closer to NYT direction (-0.13)?  "
          f"A_gap={A['gap']:+.4f} → E_gap={E['gap']:+.4f}  "
          f"{'YES' if q3 else 'NO'}")

In [ ]:

# =============================================================================
# CELL 12: Final summary
# =============================================================================

print("\n" + "=" * 78)
print("NOTEBOOK 03 COMPLETE")
print("=" * 78)
for f in sorted(GENERATIONS.glob("generations_gemini*")) + sorted(RESULTS.glob("*crossmodel*")):
    size = f.stat().st_size
    print(f"  {f.relative_to(ROOT)}  {size:,} bytes")

print("""
Interpretation guide:
- If both models YES on Q1, Q2, Q3: hierarchy generalizes → strong paper claim
- If Gemini A is already close to NYT: finding is OpenAI-specific
- If Gemini E does not help: prompt recoverability is OpenAI-specific
""")